# Lab 10 starter: tuning a random forest by cross-validation

This notebook is standalone. In the recap cell, the snapshot is loaded, the
chapter's training/test split is made, and the chapter's `error_cost` function is
defined, so that you can tune and evaluate a forest on the same customers.

The lab prompt is at the end of Chapter 10, under **Exercises**.

**Build lab.** On the training set only, compute the five-fold CV accuracy of
`RandomForestClassifier(n_estimators=400, random_state=0)` for each combination of
`max_depth` (3, 6, and `None`) and `max_features` (`"sqrt"` and 0.5). Choose the
combination with the highest CV mean accuracy. Then fit the chosen forest and
report its test accuracy once.

**Evaluate lab.** Compute the tuned forest's CV error cost at the break-even
threshold of 0.1, and compare it with the chapter's CV error costs. Then compute
its test error cost at 0.5 and 0.1 beside the two baselines. Replace the logistic
model only if the tuned forest has a lower CV error cost at 0.1 than logistic
regression. State in two sentences whether the setting chosen for accuracy also
lowered the CV error cost, and whether the logistic model is replaced under the rule.

## Running this notebook

Run the setup cell below first, then the recap cell, then work down. In Google
Colab nothing needs to be installed beyond that cell. Locally, use the
`pyba-core` environment from Appendix A.

You will enter your work in the cells marked `# TODO`. Everything else is
provided. When you are unsure what a step produced, print it: `.head()` for a
table or a column, and the bare name for a number.

**If you are in Google Colab:** this notebook opened read-only from GitHub. Click
**Copy to Drive** in the toolbar (or *File > Save a copy in Drive*) before you
edit anything, then work in that copy. It saves to your Google Drive under
*Colab Notebooks*. Edits made to the read-only original are lost when the tab
closes.

In [ ]:
# Setup. Run this cell once per session. It installs this lab's packages;
# on Google Colab it also fetches the course data.
%pip install -q pandas scikit-learn
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

## Recap: the Chapter 10 setup

The recap cell contains the chapter's code. `X` is the feature table with 20
columns, `y` is the `churned` label, and 262 of the 872 customers are held out
for testing. `error_cost(flagged, labels)` is the chapter's cost function: \$60
per offer and \$600 per churner who receives no offer. The cell prints the two
baselines on the test set, so that you can compare the forest with them in Step 5.

The chapter's cross-validated error costs at the threshold of 0.1, on the 610
training customers, are listed here for Step 4:

| Model | CV error cost at 0.1 |
|---|---|
| logistic regression | \$24,120 |
| KNN (k=15) | \$34,260 |
| tree (depth 2) | \$27,420 |
| random forest (default settings) | \$28,440 |

In [ ]:
# --- Chapter 10 recap: run this cell first ---
import pandas as pd

from pyba import DATA_DIR
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.ensemble import RandomForestClassifier

snap = pd.read_csv(DATA_DIR / "pw_customer_snapshot.csv")
X = pd.get_dummies(snap.drop(columns=["customer_id", "pilot", "churned"]),
                   columns=["business_type", "region"])
y = snap["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0)


def error_cost(flagged, labels):
    """Total error cost: $60 per offer, $600 per churner without an offer."""
    tp = (flagged & (labels == 1)).sum()
    fp = (flagged & (labels == 0)).sum()
    fn = ((~flagged) & (labels == 1)).sum()
    return 60 * (tp + fp) + 600 * fn


print(f"test set, no offers:               ${600 * y_test.sum():,}")
print(f"test set, offer to every customer: ${60 * len(y_test):,}")

## Step 1: Compute the CV accuracy of six forest settings

The six settings are provided in `settings`, one pair of `max_depth` and
`max_features` per entry. Loop over the pairs with `for depth, features in settings:`,
as in the chapter's loop over pairs in the section on how a split is chosen. In each
pass, create `RandomForestClassifier(n_estimators=400, max_depth=depth,
max_features=features, random_state=0)` and compute its five-fold CV accuracy with
`cross_val_score(model, X_train, y_train, cv=5)`, as in the chapter's section on
cross-validation. Append one row per setting: the depth, the features setting, the
mean, and the standard deviation of the five fold scores.

Use the training set only. In the table, `None` is shown as NaN, the pandas marker
for a missing number. Running the cell may take up to a minute, since
6 × 5 = 30 forests of 400 trees are fitted.

In [ ]:
settings = [(3, "sqrt"), (3, 0.5), (6, "sqrt"), (6, 0.5), (None, "sqrt"), (None, 0.5)]

# TODO: one row per setting: [depth, features, CV mean, CV std]
rows = []

# --- Look: the CV table, one row per setting ---
cv_table = pd.DataFrame(rows, columns=["max_depth", "max_features", "CV mean", "CV std"]).round(3)
cv_table

## Step 2: Choice of setting

Use this rule, the same one the chapter uses for the tree depth:

1. In the table from Step 1, find the setting with the highest CV mean and the
   setting with the second-highest CV mean.
2. If the difference between the two CV means is smaller than the highest
   setting's CV standard deviation, the two settings are practically tied. Choose
   the less flexible of the two: the one with the smaller `max_depth` (`None`
   means no limit, the most flexible), or, if the depths are equal, the one with
   `"sqrt"`.
3. Otherwise, choose the setting with the highest CV mean.

Two sentences. Name the two settings with their CV means, and give the highest
setting's standard deviation. Then state whether the two are tied, and which
setting you chose.

---

**Chosen setting:** *replace with your answer*

**Reason:** *replace with your answer*

---

## Step 3: Fit the chosen forest and report its test accuracy

Create the forest with your chosen `max_depth` and `max_features` (and
`n_estimators=400`, `random_state=0`), fit it on `X_train` and `y_train`, and print
its test accuracy with `.score`, as in the chapter's forest cell. Name the
fitted model `best_forest`.

The test set is used here for the first time in this lab. Report the number once,
and do not go back to Step 2 to change the setting.

In [ ]:
# TODO: fit the chosen forest on the training set and print its test accuracy
best_forest = None

## Step 4 (Evaluate lab): Compute the CV error cost of the tuned forest

Follow the chapter's section comparing the four models. Pass an unfitted forest with
your chosen setting to `cross_val_predict` with `cv=5` and
`method="predict_proba"`, and keep column 1, the churn probabilities, with `[:, 1]`.
Then print `error_cost(proba_cv >= 0.1, y_train)`.

Each of the 610 probabilities is computed by a forest fitted on the four folds that
do not contain that customer. Compare the result with the table in the recap.

In [ ]:
# TODO: cross-validated churn probabilities and the CV error cost at 0.1
proba_cv = None

## Step 5: Compute the test error cost at 0.5 and at 0.1

Keep the tuned forest's churn probabilities for the test customers with
`best_forest.predict_proba(X_test)[:, 1]`. Then compute
`error_cost(proba >= threshold, y_test)` for the thresholds 0.5 and 0.1. Put the
two costs in a table beside the two baselines printed by the recap cell, one row
per policy, as in the chapter's evaluation section.

In [ ]:
# TODO: test error cost of the tuned forest at 0.5 and 0.1, beside the two baselines

## Step 6: Recommendation

Use this rule: replace the logistic model only if the tuned forest has a lower CV
error cost at 0.1 than logistic regression (\$24,120). The test-set costs from
Step 5 are reported but are not part of the rule.

Two sentences. First, state whether the setting chosen for accuracy also lowered
the CV error cost, compared with the default forest (\$28,440) and with logistic
regression. Second, state the test cost of the tuned forest at 0.1 beside the two
baselines, and whether the logistic model is replaced under the rule.

---

**Effect of tuning on cost:** *replace with your answer*

**Recommendation:** *replace with your answer*

---

## Submission

Run **Restart and run all** and confirm the notebook executes top to bottom with no
errors. Then download it as `.ipynb` and upload it to the Lab 10 dropbox.

If you used an AI assistant, add a one-line note naming the tool and what it
generated, per the course policy in Appendix D.